[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/GCP_Lakehouse.ipynb)

# GCP Lakehouse — Delta Lake on GCS + BigLake + BigQuery

**Build a lakehouse on Google Cloud: write Delta tables to GCS, query them from BigQuery via BigLake, use MERGE, time travel, VACUUM, and schema evolution — all on real cloud infrastructure.**

Playbook: [Lakehouse Formats](../../playbooks/data/lakehouse-formats/README.md)

---

## What You Will Do

1. Write a Delta Lake table to GCS using PySpark on Dataproc
2. Create a BigLake external table so BigQuery can query it
3. MERGE — update records in the Delta table
4. Time travel — query previous versions from BigQuery
5. Schema evolution — handle a new column from the source
6. VACUUM — clean up old files in GCS
7. Compare: Delta Lake on GCS vs BigQuery native

## Architecture

```
Source Data --> PySpark (Dataproc) --> Delta Table (GCS) --> BigLake --> BigQuery SQL
```

**Time:** ~30 minutes

**Prerequisites:** GCP project with billing enabled. Familiarity with the [GCP Full Pipeline](GCP_Full_Pipeline.ipynb) notebook.

**Cost:** ~$1-2 (Dataproc cluster runs for ~15 minutes, GCS storage is negligible, BigQuery queries are small)

---

## 1. Configuration

In [ ]:
# === CONFIGURATION ===
# WHY: All GCP resources need a project and location.
# Change PROJECT_ID to your own project.

PROJECT_ID = "your-project-id"  # <-- CHANGE THIS
LOCATION = "us-central1"
BUCKET_NAME = f"lakehouse-{PROJECT_ID}"

# Delta table paths in GCS
DELTA_PATH = f"gs://{BUCKET_NAME}/silver/calls_delta"

# BigQuery dataset for BigLake external tables
BQ_DATASET = "lakehouse"

# Dataproc cluster
CLUSTER_NAME = "lakehouse-cluster"

print(f"Project:    {PROJECT_ID}")
print(f"Bucket:     gs://{BUCKET_NAME}/")
print(f"Delta path: {DELTA_PATH}")
print(f"BQ dataset: {BQ_DATASET}")
print(f"Cluster:    {CLUSTER_NAME}")

---

## 2. Authenticate

In [ ]:
# === AUTHENTICATE ===
import os
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated via Colab.")
else:
    print("Running locally. Use 'gcloud auth login' if not authenticated.")

!gcloud config set project {PROJECT_ID}

---

## 3. Create Infrastructure

Create the GCS bucket and BigQuery dataset.

In [ ]:
# === CREATE BUCKET ===
# WHY: Delta tables are stored as Parquet files + _delta_log in GCS.
# This bucket is the "storage layer" of the lakehouse.

!gcloud storage buckets create gs://{BUCKET_NAME} \
    --location={LOCATION} \
    --uniform-bucket-level-access \
    2>/dev/null || echo "Bucket already exists"

print(f"Bucket ready: gs://{BUCKET_NAME}/")

In [ ]:
# === CREATE BIGQUERY DATASET ===
# WHY: BigLake external tables live in a BigQuery dataset,
# but the data stays in GCS. BigQuery reads from GCS at query time.

!bq mk --dataset --location={LOCATION} {PROJECT_ID}:{BQ_DATASET} \
    2>/dev/null || echo "Dataset already exists"

print(f"Dataset ready: {BQ_DATASET}")

---

## 4. Create Sample Data

Upload call center data to GCS as the Bronze layer (raw source data).

In [ ]:
import json

# WHY: This simulates raw data arriving in the Bronze layer.
# In production, this comes from Cloud Functions, Datastream (CDC),
# or file drops from source systems.

calls = [
    {"call_id": "C-001", "customer_id": "CUST-100", "status": "in-progress",
     "duration": 0, "campaign_id": "CAMP-A",
     "call_date": "2026-04-13", "created_at": "2026-04-13T09:00:00",
     "updated_at": "2026-04-13T09:00:00"},
    {"call_id": "C-002", "customer_id": "CUST-101", "status": "resolved",
     "duration": 340, "campaign_id": "CAMP-A",
     "call_date": "2026-04-13", "created_at": "2026-04-13T09:15:00",
     "updated_at": "2026-04-13T09:18:40"},
    {"call_id": "C-003", "customer_id": "CUST-102", "status": "missed",
     "duration": 0, "campaign_id": "CAMP-B",
     "call_date": "2026-04-13", "created_at": "2026-04-13T09:22:00",
     "updated_at": "2026-04-13T09:22:00"},
    {"call_id": "C-004", "customer_id": "CUST-103", "status": "resolved",
     "duration": 480, "campaign_id": "CAMP-B",
     "call_date": "2026-04-13", "created_at": "2026-04-13T09:30:00",
     "updated_at": "2026-04-13T09:38:00"},
    {"call_id": "C-005", "customer_id": "CUST-104", "status": "voicemail",
     "duration": 60, "campaign_id": "CAMP-A",
     "call_date": "2026-04-13", "created_at": "2026-04-13T09:45:00",
     "updated_at": "2026-04-13T09:46:00"},
]

# Write as newline-delimited JSON (one JSON object per line)
with open("/tmp/calls_bronze.json", "w") as f:
    for call in calls:
        f.write(json.dumps(call) + "\n")

# Upload to GCS Bronze
!gcloud storage cp /tmp/calls_bronze.json gs://{BUCKET_NAME}/bronze/calls/

print(f"Uploaded {len(calls)} call records to gs://{BUCKET_NAME}/bronze/calls/")

**You Should See:** 5 records uploaded to the Bronze layer in GCS.

---

## 5. Write Delta Table with Dataproc (PySpark)

Now the key step: read Bronze (JSON in GCS), transform to Silver, and write as a **Delta table** back to GCS.

### Why Dataproc?

Delta Lake requires PySpark. BigQuery SQL can't write Delta tables — it can only read them (via BigLake). So we use Dataproc (managed Spark) for the write path.

```
GCS Bronze (JSON) --> Dataproc PySpark --> GCS Silver (Delta) --> BigQuery BigLake (read)
```

### Step 5a: Write the PySpark Script

In [ ]:
# === PYSPARK SCRIPT: Write Bronze to Delta Silver ===
# WHY: This script runs on Dataproc. It reads raw JSON from GCS,
# cleans the data, and writes it as a Delta table back to GCS.

spark_script = f'''
# delta_silver_transform.py
# Runs on Dataproc. Reads Bronze JSON, writes Delta Silver.

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# WHY: Delta Lake needs these Spark configurations.
# Dataproc 2.1+ includes Delta Lake JARs, but we need to
# activate the extensions.
spark = (
    SparkSession.builder
    .appName("bronze-to-delta-silver")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

# --- Read Bronze ---
bronze_path = "gs://{BUCKET_NAME}/bronze/calls/"
raw_df = spark.read.json(bronze_path)
print(f"Bronze: {{raw_df.count()}} records")

# --- Transform to Silver ---
# Clean, type-cast, add ingestion timestamp
silver_df = (
    raw_df
    .withColumn("duration", F.col("duration").cast("int"))
    .withColumn("call_date", F.to_date("call_date"))
    .withColumn("created_at", F.to_timestamp("created_at"))
    .withColumn("updated_at", F.to_timestamp("updated_at"))
    .withColumn("ingested_at", F.current_timestamp())
    .dropDuplicates(["call_id"])
)

# --- Write as Delta ---
# WHY: format("delta") instead of format("parquet").
# This creates the _delta_log transaction log alongside the Parquet files.
delta_path = "{DELTA_PATH}"
silver_df.write \
    .format("delta") \
    .partitionBy("call_date") \
    .mode("overwrite") \
    .save(delta_path)

print(f"Delta table written to {{delta_path}}")
print(f"Silver: {{silver_df.count()}} records")

# Verify: show the Delta table
spark.read.format("delta").load(delta_path).show()

spark.stop()
'''

# Save the script locally
with open("/tmp/delta_silver_transform.py", "w") as f:
    f.write(spark_script)

# Upload to GCS (Dataproc reads scripts from GCS)
!gcloud storage cp /tmp/delta_silver_transform.py gs://{BUCKET_NAME}/code/

print("PySpark script uploaded to GCS")

### Step 5b: Create Dataproc Cluster

We need a Dataproc cluster with Delta Lake support. Image version 2.1+ includes Delta Lake JARs.

In [ ]:
# === CREATE DATAPROC CLUSTER ===
# WHY: Dataproc is managed Spark. You create a cluster, submit jobs,
# and delete the cluster when done. You only pay while it runs.
#
# image-version 2.1: includes Delta Lake support
# num-workers 2: minimum for a real Spark cluster
# This takes 2-3 minutes.

!gcloud dataproc clusters create {CLUSTER_NAME} \
    --region={LOCATION} \
    --image-version=2.1-debian11 \
    --num-workers=2 \
    --worker-machine-type=n1-standard-2 \
    --master-machine-type=n1-standard-2 \
    --properties=spark:spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension,spark:spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog \
    --optional-components=DOCKER \
    2>&1 | tail -5

print(f"\nCluster {CLUSTER_NAME} is ready.")

**You Should See:** Cluster creation output ending with the cluster status. Takes 2-3 minutes.

### Step 5c: Submit the Delta Transform Job

In [ ]:
# === SUBMIT PYSPARK JOB ===
# WHY: The PySpark script runs on the Dataproc cluster, reads
# Bronze from GCS, and writes the Delta table to GCS.

!gcloud dataproc jobs submit pyspark \
    gs://{BUCKET_NAME}/code/delta_silver_transform.py \
    --cluster={CLUSTER_NAME} \
    --region={LOCATION} \
    2>&1 | tail -20

print("\nDelta Silver transform complete.")

In [ ]:
# === VERIFY: Check Delta table structure in GCS ===
# WHY: A Delta table has Parquet data files + a _delta_log directory.
# The _delta_log is what makes it a Delta table, not just Parquet files.

print("Delta table structure in GCS:")
print("="*50)
!gcloud storage ls gs://{BUCKET_NAME}/silver/calls_delta/

print("\nTransaction log:")
!gcloud storage ls gs://{BUCKET_NAME}/silver/calls_delta/_delta_log/

**You Should See:**
- Partition directories (`call_date=2026-04-13/`)
- A `_delta_log/` directory with `00000000000000000000.json` (commit 0)

This is **Version 0** of the Delta table on GCS.

---

## 6. BigLake — Query Delta from BigQuery

Now the lakehouse magic: query the Delta table in GCS using standard BigQuery SQL. No data copying. BigQuery reads directly from GCS.

### What is BigLake?

BigLake is GCP's bridge between cloud storage and BigQuery. It creates an **external table** that points to files in GCS. When you query the table, BigQuery reads from GCS at query time.

**Analogy:** A library catalog card that points to a book on a different shelf. The catalog (BigQuery) knows where the book is (GCS). When you ask for the book, the librarian fetches it from the correct shelf.

### AWS Equivalent

| GCP | AWS |
|---|---|
| BigLake external table | Athena + Glue Catalog |
| BigQuery reads from GCS | Athena reads from S3 |
| `CREATE EXTERNAL TABLE` | `CREATE EXTERNAL TABLE` (in Athena) |

In [ ]:
# === CREATE BIGLAKE CONNECTION ===
# WHY: BigLake needs a "connection" to read from GCS.
# The connection has a service account that BigQuery uses to access GCS.

CONNECTION_NAME = "biglake-gcs"

!bq mk --connection \
    --connection_type=CLOUD_RESOURCE \
    --location={LOCATION} \
    {CONNECTION_NAME} \
    2>/dev/null || echo "Connection already exists"

# Get the service account created for this connection
import subprocess, json as j

result = subprocess.run(
    ["bq", "show", "--connection", f"{PROJECT_ID}.{LOCATION}.{CONNECTION_NAME}",
     "--format=json"],
    capture_output=True, text=True
)

if result.returncode == 0:
    conn_info = j.loads(result.stdout)
    sa_email = conn_info.get("cloudResource", {}).get("serviceAccountId", "")
    print(f"BigLake service account: {sa_email}")
else:
    print("Could not retrieve connection info. Check that the connection was created.")
    sa_email = ""

In [ ]:
# === GRANT GCS ACCESS TO BIGLAKE SERVICE ACCOUNT ===
# WHY: The BigLake connection's service account needs permission
# to read the Delta table files in GCS.

if sa_email:
    !gcloud storage buckets add-iam-policy-binding gs://{BUCKET_NAME} \
        --member=serviceAccount:{sa_email} \
        --role=roles/storage.objectViewer \
        2>&1 | tail -3
    print(f"Granted GCS read access to {sa_email}")
else:
    print("No service account found. Grant access manually in the console.")

In [ ]:
# === CREATE BIGLAKE EXTERNAL TABLE ===
# WHY: This tells BigQuery "there's a Delta table at this GCS path.
# When I query this table, read the Delta log to find the current
# Parquet files and return the data."
#
# The data stays in GCS. BigQuery doesn't copy it.
# You pay GCS storage rates ($0.02/GB), not BigQuery storage rates.

create_table_sql = f"""
CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.{BQ_DATASET}.calls_delta`
WITH CONNECTION `{PROJECT_ID}.{LOCATION}.{CONNECTION_NAME}`
OPTIONS (
    format = 'DELTA_LAKE',
    uris = ['{DELTA_PATH}']
);
"""

!bq query --use_legacy_sql=false "{create_table_sql}"

print(f"BigLake external table created: {BQ_DATASET}.calls_delta")

In [ ]:
# === QUERY: Read Delta table from BigQuery ===
# WHY: This is the payoff. Standard BigQuery SQL reads a Delta table
# stored in GCS. No Spark needed for reading.

!bq query --use_legacy_sql=false \
    'SELECT call_id, customer_id, status, duration, call_date FROM `{PROJECT_ID}.{BQ_DATASET}.calls_delta` ORDER BY call_id'

**You Should See:** 5 call records. C-001 has `status=in-progress`. This is the Delta table on GCS being queried through BigQuery SQL.

---

## 7. MERGE — Update Records in Delta

Call C-001 got resolved (duration: 480 seconds). We need to update it in the Delta table.

**Key point:** MERGE runs in PySpark on Dataproc (not BigQuery). BigQuery can READ Delta tables but cannot WRITE to them. The write path always goes through Spark.

In [ ]:
# === PYSPARK SCRIPT: MERGE updated calls into Delta ===

merge_script = f'''
# delta_merge.py
# MERGE incoming changes into the Delta Silver table.

from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime

spark = (
    SparkSession.builder
    .appName("delta-merge")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

delta_path = "{DELTA_PATH}"

# Incoming update: C-001 is now resolved, plus a new call C-006
incoming_data = [
    Row(call_id="C-001", customer_id="CUST-100", status="resolved",
        duration=480, campaign_id="CAMP-A",
        call_date="2026-04-13",
        created_at="2026-04-13T09:00:00",
        updated_at="2026-04-13T09:08:00"),
    Row(call_id="C-006", customer_id="CUST-105", status="resolved",
        duration=200, campaign_id="CAMP-C",
        call_date="2026-04-13",
        created_at="2026-04-13T10:00:00",
        updated_at="2026-04-13T10:03:20"),
]

incoming_df = spark.createDataFrame(incoming_data)

# Cast types to match Delta table schema
incoming_df = (
    incoming_df
    .withColumn("duration", F.col("duration").cast("int"))
    .withColumn("call_date", F.to_date("call_date"))
    .withColumn("created_at", F.to_timestamp("created_at"))
    .withColumn("updated_at", F.to_timestamp("updated_at"))
    .withColumn("ingested_at", F.current_timestamp())
)

# Load the existing Delta table
delta_table = DeltaTable.forPath(spark, delta_path)

# MERGE: update if call_id matches, insert if new
(
    delta_table.alias("target")
    .merge(
        incoming_df.alias("source"),
        "target.call_id = source.call_id AND target.call_date = source.call_date"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE complete!")
print(f"Table now has {{spark.read.format('delta').load(delta_path).count()}} records")
spark.read.format("delta").load(delta_path).orderBy("call_id").show()

# Show history
print("Delta table history:")
delta_table.history().select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

spark.stop()
'''

with open("/tmp/delta_merge.py", "w") as f:
    f.write(merge_script)

!gcloud storage cp /tmp/delta_merge.py gs://{BUCKET_NAME}/code/
print("MERGE script uploaded")

In [ ]:
# === SUBMIT MERGE JOB ===

!gcloud dataproc jobs submit pyspark \
    gs://{BUCKET_NAME}/code/delta_merge.py \
    --cluster={CLUSTER_NAME} \
    --region={LOCATION} \
    2>&1 | tail -25

print("\nMERGE job complete.")

In [ ]:
# === VERIFY: Query updated Delta table from BigQuery ===
# WHY: BigLake automatically picks up the new Delta version.
# No need to recreate the external table.

!bq query --use_legacy_sql=false \
    'SELECT call_id, status, duration FROM `{PROJECT_ID}.{BQ_DATASET}.calls_delta` ORDER BY call_id'

**You Should See:** 6 records. C-001 is now `resolved` with `duration=480`. C-006 is new. This is **Version 1** of the Delta table.

---

## 8. Time Travel

Query previous versions of the Delta table. This runs through PySpark (BigLake doesn't support Delta time travel directly yet).

In [ ]:
# === PYSPARK SCRIPT: Time Travel ===

time_travel_script = f'''
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("delta-time-travel")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

delta_path = "{DELTA_PATH}"

# Version 0: Before the MERGE
print("=== Version 0 (before MERGE) ===")
v0 = spark.read.format("delta").option("versionAsOf", 0).load(delta_path)
v0.filter("call_id = 'C-001'").select("call_id", "status", "duration").show()
print(f"Total records in Version 0: {{v0.count()}}")

# Version 1: After the MERGE
print("\n=== Version 1 (after MERGE) ===")
v1 = spark.read.format("delta").option("versionAsOf", 1).load(delta_path)
v1.filter("call_id = 'C-001'").select("call_id", "status", "duration").show()
print(f"Total records in Version 1: {{v1.count()}}")

# Compare: what changed?
from pyspark.sql import functions as F
changed = (
    v0.alias("before")
    .join(v1.alias("after"), "call_id")
    .filter("before.status != after.status OR before.duration != after.duration")
    .select(
        "call_id",
        F.col("before.status").alias("old_status"),
        F.col("after.status").alias("new_status"),
        F.col("before.duration").alias("old_duration"),
        F.col("after.duration").alias("new_duration"),
    )
)
print("\n=== Records that changed ===")
changed.show()

spark.stop()
'''

with open("/tmp/delta_time_travel.py", "w") as f:
    f.write(time_travel_script)

!gcloud storage cp /tmp/delta_time_travel.py gs://{BUCKET_NAME}/code/

!gcloud dataproc jobs submit pyspark \
    gs://{BUCKET_NAME}/code/delta_time_travel.py \
    --cluster={CLUSTER_NAME} \
    --region={LOCATION} \
    2>&1 | tail -30

**You Should See:**
- Version 0: C-001 = in-progress, 5 records total
- Version 1: C-001 = resolved, 6 records total
- Changed records: C-001 (in-progress → resolved, 0 → 480)

### BigQuery Native Time Travel (for comparison)

BigQuery has built-in time travel, but only for native tables (not BigLake external tables):

```sql
-- BigQuery native time travel (works on native tables only)
SELECT * FROM silver.calls
FOR SYSTEM_TIME AS OF TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 HOUR);
```

BigQuery time travel is limited to 7 days. Delta Lake time travel is limited by your VACUUM retention policy.

---

## 9. Schema Evolution

The source system starts sending a new field: `agent_language`. The Delta table doesn't have this column yet.

In [ ]:
# === PYSPARK SCRIPT: Schema Evolution ===

schema_script = f'''
from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("delta-schema-evolution")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

delta_path = "{DELTA_PATH}"

# New data with extra column: agent_language
new_data = spark.createDataFrame([
    Row(call_id="C-007", customer_id="CUST-106", status="resolved",
        duration=300, campaign_id="CAMP-B",
        call_date="2026-04-14",
        created_at="2026-04-14T09:00:00",
        updated_at="2026-04-14T09:05:00",
        agent_language="Spanish"),
])

new_data = (
    new_data
    .withColumn("duration", F.col("duration").cast("int"))
    .withColumn("call_date", F.to_date("call_date"))
    .withColumn("created_at", F.to_timestamp("created_at"))
    .withColumn("updated_at", F.to_timestamp("updated_at"))
    .withColumn("ingested_at", F.current_timestamp())
)

# WHY: mergeSchema=true tells Delta to add the new column.
# Without it, the write FAILS (Delta protects against schema changes by default).
new_data.write \
    .format("delta") \
    .option("mergeSchema", "true") \
    .mode("append") \
    .save(delta_path)

print("Schema evolution complete: agent_language column added")
print(f"\nUpdated schema:")
spark.read.format("delta").load(delta_path).printSchema()

print("\nAll records (old records have null for agent_language):")
spark.read.format("delta").load(delta_path) \
    .select("call_id", "status", "duration", "agent_language") \
    .orderBy("call_id").show()

spark.stop()
'''

with open("/tmp/delta_schema_evolution.py", "w") as f:
    f.write(schema_script)

!gcloud storage cp /tmp/delta_schema_evolution.py gs://{BUCKET_NAME}/code/

!gcloud dataproc jobs submit pyspark \
    gs://{BUCKET_NAME}/code/delta_schema_evolution.py \
    --cluster={CLUSTER_NAME} \
    --region={LOCATION} \
    2>&1 | tail -20

In [ ]:
# === VERIFY: BigQuery sees the new column ===
# WHY: BigLake picks up schema changes automatically.
# But you may need to refresh the external table definition.

# Recreate the external table to pick up the schema change
!bq query --use_legacy_sql=false "{create_table_sql}"

!bq query --use_legacy_sql=false \
    'SELECT call_id, status, duration, agent_language FROM `{PROJECT_ID}.{BQ_DATASET}.calls_delta` ORDER BY call_id'

**You Should See:** 7 records. C-007 has `agent_language=Spanish`. All other records have `null` for that column.

---

## 10. VACUUM — Clean Up Old Files

In [ ]:
# === PYSPARK SCRIPT: VACUUM ===

vacuum_script = f'''
from pyspark.sql import SparkSession
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("delta-vacuum")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

delta_path = "{DELTA_PATH}"
delta_table = DeltaTable.forPath(spark, delta_path)

# Show full history
print("Delta table history (all versions):")
delta_table.history().select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

# Show table detail (file count)
print("\nTable detail:")
spark.sql(f"DESCRIBE DETAIL delta.`{{delta_path}}`").select("numFiles", "sizeInBytes").show()

# VACUUM: remove files older than 0 hours (demo only)
# WHY: In production, use retentionHours=168 (7 days).
# retentionHours=0 deletes ALL old files — no time travel after this.
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
delta_table.vacuum(retentionHours=0)

print("\nVACUUM complete. Old files removed from GCS.")
print("Table detail after VACUUM:")
spark.sql(f"DESCRIBE DETAIL delta.`{{delta_path}}`").select("numFiles", "sizeInBytes").show()

spark.stop()
'''

with open("/tmp/delta_vacuum.py", "w") as f:
    f.write(vacuum_script)

!gcloud storage cp /tmp/delta_vacuum.py gs://{BUCKET_NAME}/code/

!gcloud dataproc jobs submit pyspark \
    gs://{BUCKET_NAME}/code/delta_vacuum.py \
    --cluster={CLUSTER_NAME} \
    --region={LOCATION} \
    2>&1 | tail -25

**You Should See:** File count decreases after VACUUM. Old Parquet files from Version 0 are deleted from GCS.

---

## 11. Compare: Delta on GCS vs BigQuery Native

Both approaches work on GCP. Here's when to pick each.

In [ ]:
# WHY: This isn't code — it's a decision framework you should internalize.

comparison = """
╔══════════════════════════════╦══════════════════════════════╦══════════════════════════════╗
║ Feature                      ║ BigQuery Native              ║ Delta Lake on GCS + BigLake  ║
╠══════════════════════════════╬══════════════════════════════╬══════════════════════════════╣
║ ACID transactions            ║ Yes                          ║ Yes                          ║
║ MERGE                        ║ Yes (SQL)                    ║ Yes (PySpark)                ║
║ Time travel                  ║ 7 days max                   ║ Configurable (VACUUM policy) ║
║ Schema evolution             ║ Add columns                  ║ Add columns + mergeSchema    ║
║ Storage cost                 ║ $0.02/GB (active)            ║ $0.02/GB (GCS standard)      ║
║ Multi-engine access          ║ BigQuery only                ║ Spark, Trino, Flink, BigQuery║
║ Write path                   ║ SQL or streaming API         ║ PySpark (needs Dataproc)     ║
║ Read path                    ║ SQL                          ║ SQL (via BigLake) or Spark   ║
║ Open format (no lock-in)     ║ No (proprietary)             ║ Yes (Parquet + open metadata)║
║ Operational complexity       ║ Low (managed)                ║ Higher (manage Dataproc)     ║
╠══════════════════════════════╬══════════════════════════════╬══════════════════════════════╣
║ PICK THIS WHEN               ║ All analytics in BigQuery.   ║ Need multi-engine, open      ║
║                              ║ Don't need multi-engine.     ║ format, or Spark processing. ║
║                              ║ Want simplicity.             ║ Want vendor independence.    ║
╚══════════════════════════════╩══════════════════════════════╩══════════════════════════════╝
"""

print(comparison)

---

## 12. Cleanup

**Delete the Dataproc cluster to stop paying.** GCS and BigQuery resources cost very little at this scale.

In [ ]:
# === DELETE DATAPROC CLUSTER ===
# WHY: The cluster costs ~$0.30/hour. Delete it when you're done.
# The Delta table on GCS and the BigLake table in BigQuery persist.

!gcloud dataproc clusters delete {CLUSTER_NAME} \
    --region={LOCATION} \
    --quiet

print(f"Cluster {CLUSTER_NAME} deleted. You're no longer paying for compute.")
print(f"\nYour Delta table is still at: {DELTA_PATH}")
print(f"Your BigLake table is still queryable: {BQ_DATASET}.calls_delta")

In [ ]:
# === OPTIONAL: Full cleanup (delete everything) ===
# Uncomment these lines to remove all resources created in this notebook.

# Delete BigQuery dataset and all tables
# !bq rm -r -f {PROJECT_ID}:{BQ_DATASET}

# Delete BigLake connection
# !bq rm --connection {PROJECT_ID}.{LOCATION}.{CONNECTION_NAME}

# Delete GCS bucket and all contents
# !gcloud storage rm --recursive gs://{BUCKET_NAME}/

# print("All resources deleted.")

---

## What You Built

```
Bronze (JSON in GCS)
    │
    ▼
PySpark on Dataproc
    │
    ▼
Delta Table (Parquet + _delta_log in GCS)  ◄── MERGE, schema evolution, VACUUM
    │
    ▼
BigLake External Table
    │
    ▼
BigQuery SQL  ◄── Standard queries, joins, dashboards
```

**Key takeaway:** The data stays in GCS (cheap, open format). BigQuery reads it on demand (no copying). Spark writes it (MERGE, schema evolution). This is the lakehouse pattern on GCP.

---

## AWS Equivalent

| This Notebook (GCP) | AWS Equivalent |
|---|---|
| GCS | S3 |
| Dataproc | EMR |
| BigLake external table | Glue Catalog + Athena |
| BigQuery SQL | Athena SQL |
| `format = 'DELTA_LAKE'` | Athena supports Delta and Iceberg natively |

The same Delta table on S3 can be queried through Athena with no code changes to the data.

---

## Next Steps

- [Delta Lake Hello World](Delta_Lake_Hello_World.ipynb) — local PySpark, no cloud needed (for learning the API)
- [Lakehouse Formats - How It Works](../../playbooks/data/lakehouse-formats/04_How_It_Works.md) — Delta transaction log internals
- [Lakehouse Formats - Production Patterns](../../playbooks/data/lakehouse-formats/06_Production_Patterns.md) — OPTIMIZE, Z-ORDER, concurrent writes
- [Lakehouse Formats - Decision Guide](../../playbooks/data/lakehouse-formats/10_Decision_Guide.md) — Delta vs Iceberg vs BigQuery native

---

[Community](https://www.skool.com/deliverymomentum) | [Book a 1:1](https://calendly.com/sunil-mogadati/connect)